In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import kalepy as kale

import holodeck as holo
import holodeck.sams
from holodeck import utils, plot
from holodeck.constants import MSOL


See [Leja+2020 - A New Census of the 0.2 < z < 3.0 Universe. I. The Stellar Mass Function ](https://ui.adsabs.harvard.edu/abs/2020ApJ...893..111L/abstract)
This notebook converts from the "anchor point" fits, into fits of the redshift-function expansion.  i.e. converting from the parameters given in [Leja+2020] Fig.3, into the coefficients for their Eq.17.

See [Leja+2020 - A New Census of the 0.2 < z < 3.0 Univers. I. The Stellar Mass Function ](https://ui.adsabs.harvard.edu/abs/2020ApJ...893..111L/abstract)
This notebook convers from the "anchor point" fits, into fits of the redshift-function expansion. I.e., it converts from the parameters given in [Leja+2020] Fig. 3 into the coefficients for their Eq. 17.

# GSMF

## the function itself

In [ ]:
gsmf = holo.sams.components.GSMF_Double_Schechter()

mstar = np.logspace(8, 12, 100) * MSOL

for redz in [0.1, 0.5, 1.5, 3.0]:

    phi = gsmf(mstar, redz)
    cc, = plt.loglog(mstar, phi, label=redz)
    cc = cc.get_color()

    phi = gsmf._gsmf_one(mstar, redz)
    plt.loglog(mstar, phi, ls='--', color=cc, alpha=0.4)
    phi = gsmf._gsmf_two(mstar, redz)
    plt.loglog(mstar, phi, ls='--', color=cc, alpha=0.4)

ax = plt.gca()
ylim = [1e-5, 1e-1]
ax.set(ylim=ylim)
ax.legend()
plt.show()

## used in SAM

In [ ]:
SHAPE = 30
NFREQ = 20
NREALS = 100
fobs_cents, fobs_edges = utils.pta_freqs(num=NFREQ)

# calculate GWB using double-schechter GSMF
gsmf_double = holo.sams.components.GSMF_Double_Schechter()
sam_double = holo.sams.Semi_Analytic_Model(gsmf=gsmf_double, shape=SHAPE)
hc_ss, hc_bg = sam_double.gwb(fobs_edges, realize=NREALS)
gwb_double = np.sqrt(hc_bg**2 + np.sum(hc_ss**2, axis=-1))

# calculate GWB using standard (single) schechter GSMF
gsmf_single = holo.sams.components.GSMF_Schechter()
sam_single = holo.sams.Semi_Analytic_Model(gsmf=gsmf_single, shape=SHAPE)
hc_ss, hc_bg = sam_single.gwb(fobs_edges, realize=NREALS)
gwb_single = np.sqrt(hc_bg**2 + np.sum(hc_ss**2, axis=-1))

# plot both GWBs
fig, ax = plot.figax()
plot.draw_gwb(ax, fobs_cents*1e9, gwb_single, plot=dict(label='single'), nsamp=None, fracs=[0.5])
plot.draw_gwb(ax, fobs_cents*1e9, gwb_double, plot=dict(label='double'), nsamp=None, fracs=[0.5])

plot._twin_yr(ax)
ax.legend()

plt.show()


In [ ]:
# import numpy as np
# # 1. Load actual Leja 2020 anchor point data
# d = np.load('leja_2020_cov_matrix.npz') # this file is not being distributed. It is below. This commented out cell is for reference only.
# # Take only the first 11 parameters (ignoring the 12th nuisance parameter)
# leja_means = d['means'][:11]
# leja_cov = d['cov'][:11, :11]
# print(leja_means)
# print(leja_cov)

# Covariance Derivation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Redshift values for anchor points
ZVALS = [0.2, 1.6, 3.0]

def get_cs(yy):
    z1, z2, z3 = ZVALS
    if yy.ndim == 1:
        y1, y2, y3 = yy
    else:
        y1, y2, y3 = yy
    
    # calculate quadratic coefficients (c0, c1, c2)
    tt = (z1 - z3) / (z2 - z1)
    denom = (z2**2 - z1**2) * tt
    denom = z3**2 - z1**2 + denom
    numer = (y3 - y1) + (y2 - y1) * tt
    c2 = numer / denom
    c1 = y2 - y1 - c2*(z2**2 - z1**2)
    c1 /= (z2 - z1)
    c0 = y1 - c2*z1**2 - c1*z1
    return np.array([c0, c1, c2])

# The following were calculated by Kayhan Gültekin based on the samples kindly given to him by Joel Leja.
# Those samples are the samples from Leja et al. 2020.  To read in those samples directly, requies 
# a little bit of Leja's code, which was provided on an implied as-needed basis, and thus can't be packaged 
# with this MIT-license software. 
leja_means = [-2.4422406916642814, -3.080747932958568, -4.14849830446342, 
              -2.8924538255154717, -3.292752161414908, -3.514022217207719, 
              10.789569884035464, 10.884452198574706, 10.844430073314523, 
              -0.27748989730673845, -1.4791340073222423]

leja_cov = np.array([
    [ 0.00047725859706431866, -2.3819393754385858e-05,  0.0009776847787736106, -8.657865436015189e-05, -9.808981685361722e-05,  3.865079164220302e-05, -0.00019660201739055177,  5.9193312016045204e-05, -0.00017561540932040277, -2.589002616882552e-05, -8.835140274735066e-05],
    [-2.3819393754385864e-05,  0.000672825880320772,    0.0014566616014635152, -5.411251959428764e-05,  4.5329485639495556e-05,  0.00010423083635911213,  6.838502243879605e-05,  -0.00020937074048168184,  8.269646531518938e-05,  0.0001270815654737323,  -5.388850501970634e-05],
    [ 0.0009776847787736106,   0.0014566616014635152,   0.011619959253836116,  -0.00070438357377686,    -0.00046388698274285985, 0.0006463753115252177,   0.00016922205470541496, -2.5717650171719375e-05, -0.0016559793067200462,  -0.001273609877257321,   -0.0003756697453602916],
    [-8.65786543601519e-05,   -5.411251959428764e-05,  -0.0007043835737768598,  0.0012520287230058235,   0.0009931274404713646,   0.0005374759895618242,  -0.00044154799940461785, -0.00038598878630885465, -9.475809982822242e-05,  0.002212362410556788,    0.0005312581333430597],
    [-9.808981685361723e-05,   4.5329485639495556e-05, -0.0004638869827428598,  0.0009931274404713646,   0.0009009778771993757,   0.00041174334102463096, -0.0002973952076267754,  -0.0004022391454127791,   2.2471717994142344e-06, 0.0018985020959559364,   0.0004429897579289962],
    [ 3.865079164220302e-05,   0.00010423083635911213,  0.0006463753115252177,   0.0005374759895618242,   0.00041174334102463096,  0.0008380369193863015,  -0.00021809980984234526, -0.00017078771318938165, -0.0005779375831214545,   0.0008527164750068883,   0.0002008805175322571],
    [-0.00019660201739055177,  6.838502243879605e-05,   0.00016922205470541502, -0.00044154799940461785, -0.0002973952076267754,  -0.00021809980984234526,  0.0003582748899412982,   0.00011110594347859527,  0.00021229300634841583, -0.0008783846106115477,  -0.0001410459405525553],
    [ 5.9193312016045204e-05, -0.00020937074048168184, -2.571765017171938e-05,  -0.00038598878630885465, -0.0004022391454127791,  -0.00017078771318938165,  0.00011110594347859527,  0.00034423650061243095,  0.00013532309875688612, -0.0009281610162858943,  -0.0001618095598127289],
    [-0.00017561540932040277,  8.269646531518938e-05,  -0.0016559793067200462,  -9.475809982822242e-05,  2.247171799414231e-06,  -0.0005779375831214545,   0.00021229300634841586,  0.00013532309875688612,  0.0016641291167357646,  -0.00010508506819281946, -1.3836551160806821e-05],
    [-2.589002616882552e-05,   0.00012708156547373226, -0.0012736098772573208,  0.0022123624105567876,   0.0018985020959559364,   0.0008527164750068883,  -0.0008783846106115476,  -0.0009281610162858944,  -0.00010508506819281946,  0.004576153816388647,    0.0009059432200967123],
    [-8.835140274735067e-05,  -5.388850501970635e-05,  -0.0003756697453602916,   0.0005312581333430597,   0.0004429897579289962,   0.0002008805175322571,  -0.0001410459405525553,  -0.00016180955981272892, -1.3836551160806821e-05,  0.0009059432200967123,   0.0002465223047268773]
])

# 2. Monte Carlo propagate the full 11D uncertainties
NUM = 1000000
yy_samples = np.random.multivariate_normal(leja_means, leja_cov, size=NUM).T

# 3. Transform the first 9 anchor point parameters into quadratic coefficients
cs_samples = np.zeros_like(yy_samples)
for i in range(3):
    idx = slice(i*3, (i+1)*3)
    cs_samples[idx] = get_cs(yy_samples[idx])

# The last two parameters (alpha1, alpha2) are carried over directly
cs_samples[9:11] = yy_samples[9:11]

# 4. Compute final converted means and fully correlated covariance
final_means = np.mean(cs_samples, axis=1)
cov = np.cov(cs_samples)

# 5. Repair covariance if needed for strict positive-semidefiniteness
def repair_covariance(m):
    m = (m + m.T) / 2
    eigval, eigvec = np.linalg.eigh(m)
    eigval = np.maximum(eigval, 1e-10)
    return eigvec @ np.diag(eigval) @ eigvec.T

cov_fixed = repair_covariance(cov)

# print("Final 11D Means:\n", final_means)
# print("\nFinal Fixed Covariance Matrix (11x11):\n", repr(cov_fixed))
print("GSMF_COV_NAMES = [")
print("    'gsmf_log10_phi_one_z0', 'gsmf_log10_phi_one_z1', 'gsmf_log10_phi_one_z2',")
print("    'gsmf_log10_phi_two_z0', 'gsmf_log10_phi_two_z1', 'gsmf_log10_phi_two_z2',")
print("    'gsmf_log10_mstar_z0', 'gsmf_log10_mstar_z1', 'gsmf_log10_mstar_z2',")
print("    'gsmf_alpha_one', 'gsmf_alpha_two'")
print("]")
print(f"GSMF_COV_MEANS = {list(np.round(final_means, 4))}")
print("GSMF_COV_MATRIX = [")
for i, row in enumerate(np.round(cov_fixed, 6)):
    suffix = "," if i < len(cov_fixed)-1 else ""
    print(f"    {list(row)}{suffix}")
print("]")

## Visualizing the Fits

In [ ]:
def use_cs(zz, cc):
    return cc[0] + cc[1]*zz + cc[2]*(zz**2)

fig, ax = plt.subplots()
redz = np.linspace(0.0, 5.0, 100)
for i, par in enumerate(['logphi1', 'logphi2', 'logmstar']):
    par_cs = final_means[i*3 : (i+1)*3]
    yy = use_cs(redz, par_cs)
    ax.plot(redz, yy, label=par)
    # Plot the original anchor points for comparison
    ax.scatter(ZVALS, pars[par], color=f"C{i}")

ax.set_xlabel('Redshift $z$')
ax.set_ylabel('Parameter Value')
ax.legend()
plt.show()


## Consistency and Distribution Check
We now check the distributions of the derived coefficients to ensure they are approximately Gaussian and centered correctly.

In [ ]:
import kalepy as kale
NUM = 1e5

# This checks that our sampling correctly recovers the anchor points and 
# visualizes the distribution of the derived quadratic coefficients.
fig, axes = plt.subplots(figsize=[15, 12], ncols=3, nrows=3)

for jj, par in enumerate(['logphi1', 'logphi2', 'logmstar']):
    axrow = axes[jj]

    yave = np.array(pars[par])
    yerr = np.array(pars[par + "_err"])

    # Sample from the anchor points
    yy = np.random.normal(yave[:, np.newaxis], yerr[:, np.newaxis], size=(3, int(NUM)))

    # Get the coefficients for all samples
    cs = get_cs(yy)
    cave = get_cs(yave)

    for ii, ax in enumerate(axrow):
        title = f"{par:>10s} : c{ii}"
        ax.set_title(title)
        ax.axvline(cave[ii], color='k', ls='--', alpha=0.25)

        # Plot density
        aa, bb = kale.density(cs[ii], probability=True)
        ax.plot(aa, bb, alpha=0.5, zorder=10, lw=2.0)

        # Fit a Gaussian to the sample distribution to check mean/sigma
        ave = np.mean(cs[ii])
        std = np.std(cs[ii])
        popt, pcov = holo.utils.fit_gaussian(aa, bb, guess=[bb.max(), ave, std])

        # Plot the fit
        bb_fit = holo.utils._func_gaussian(aa, *popt)
        lab = f"${popt[1]:+7.3f} \\pm {popt[2]:.3f}$"
        print(title + " -- " + lab)
        ax.plot(aa, bb_fit, alpha=0.5, ls=':', lw=3.0, label=lab)

        # Histogram for comparison
        ax.hist(cs[ii], bins=50, histtype='step', density=True, alpha=0.25, lw=2.0)

        ax.legend()

plt.tight_layout()
plt.show()


## Repairing the Covariance Matrix
Numerical propagation can sometimes result in a correlation matrix that isn't strictly positive-semidefinite. We use `repair_covariance` to ensure stability.

In [ ]:
from holodeck.utils import repair_covariance
# Repair the covariance matrix to ensure it is symmetric and PSD
# (Using the utility function we added to holodeck/utils.py)
cov_fixed = utils.repair_covariance(cov)

print("Final 11D Means:\n", list(final_means))
print("\nFinal Fixed Covariance Matrix (11x11):\n", repr(cov_fixed))

# Verification
print("\nMatrix is symmetric:", np.allclose(cov_fixed, cov_fixed.T))
print("Matrix is PSD:", np.all(np.linalg.eigvalsh(cov_fixed) > 0))


## Final Constants for `param_spaces.py`
These are the verified values for the covariant GSMF parameter space.

In [ ]:
GSMF_COV_NAMES = [
    "gsmf_log10_phi_one_z0", "gsmf_log10_phi_one_z1", "gsmf_log10_phi_one_z2",
    "gsmf_log10_phi_two_z0", "gsmf_log10_phi_two_z1", "gsmf_log10_phi_two_z2",
    "gsmf_log10_mstar_z0", "gsmf_log10_mstar_z1", "gsmf_log10_mstar_z2",
    "gsmf_alpha_one", "gsmf_alpha_two"
]
print("GSMF_COV_NAMES =", GSMF_COV_NAMES)
print("GSMF_COV_MEANS =", list(np.round(final_means, 4)))
print("GSMF_COV_MATRIX =", np.round(cov_fixed, 6).tolist())
